# fbsa_skip — Label-Efficiency Sweep

Trains `fbsa_skip` (v3: slot + encoder fusion + image-stem skip connections) at
**four data fractions: 10%, 25%, 50%, 100%** of the BRISC training set.
The validation (test) set is held constant across all runs so metrics are
directly comparable.

### What this notebook does
1. Downloads BRISC-2025 data from Kaggle.
2. Runs four independent training jobs (one per fraction) and saves each run
   under `OUT_DIR/frac_010/`, `OUT_DIR/frac_025/`, etc.
3. Plots:
   - Validation Dice learning curves (all fractions overlaid)
   - Training Dice learning curves (all fractions overlaid)
   - Loss convergence curves (all fractions overlaid)
   - Best val-Dice vs fraction (bar + line)
   - Multi-metric grid (Dice, IoU, Hausdorff) at convergence
4. Prints a summary table and saves it to CSV.

Run **only after** `01_sanity.ipynb` passes.  
Estimated wall-time on A100: ~6 h total (all four runs).

In [ ]:
import sys
from pathlib import Path

PROJECT = Path('..').resolve()
sys.path.insert(0, str(PROJECT))

%pip -q install kagglehub scipy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
ARCH       = 'fbsa_skip'   # fixed — this notebook is for fbsa_skip only
FRACTIONS  = [0.10, 0.25, 0.50, 1.00]   # data fractions to evaluate
EPOCHS     = 30
BATCH_SIZE = 16
LR         = 3e-4
NUM_SLOTS  = 2
SEED       = 42

OUT_DIR    = f'/content/drive/MyDrive/fbsa_runs/{ARCH}_label_eff'

# Set SKIP_EXISTING=True to reload saved history.pt files instead of retraining
SKIP_EXISTING = True

print(f'arch={ARCH}  fractions={FRACTIONS}  epochs={EPOCHS}')
print(f'out_dir={OUT_DIR}')

In [ ]:
import kagglehub

data_root = str(
    Path(kagglehub.dataset_download('briscdataset/brisc2025'))
    / 'brisc2025' / 'segmentation_task'
)
print('BRISC data at:', data_root)

## Train all fractions

Each fraction trains independently from a fresh random init.
Results (history + best checkpoint) are saved per-fraction under `OUT_DIR`.
If `SKIP_EXISTING=True`, already-completed runs are loaded from disk.

In [ ]:
from tumor_seg.config import TrainConfig
from tumor_seg.label_efficiency import run_label_efficiency

base_cfg = TrainConfig(
    arch=ARCH,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    num_slots=NUM_SLOTS,
    seed=SEED,
    data_root=data_root,
    out_dir=OUT_DIR,
)

results = run_label_efficiency(
    data_root=data_root,
    out_dir=OUT_DIR,
    fractions=FRACTIONS,
    base_cfg=base_cfg,
    skip_existing=SKIP_EXISTING,
)
print('\nAll fractions done.')

## (Optional) Reload results from a previous run

If you already trained all fractions in a prior session, run this cell
to reload the saved histories without retraining.

In [ ]:
# Uncomment and run this cell to reload saved results instead of retraining.

# import torch
# from pathlib import Path
#
# results = {}
# for frac in FRACTIONS:
#     pct = int(round(frac * 100))
#     hp = Path(OUT_DIR) / f'frac_{pct:03d}' / 'history.pt'
#     if hp.exists():
#         results[frac] = torch.load(hp, weights_only=False)['history']
#         print(f'Loaded frac={pct}%  ({len(results[frac])} epochs)')
#     else:
#         print(f'WARNING: {hp} not found')

## Plotting utilities

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# Consistent color palette — one colour per fraction
FRAC_COLORS = {
    0.10: '#e41a1c',   # red
    0.25: '#ff7f00',   # orange
    0.50: '#377eb8',   # blue
    1.00: '#4daf4a',   # green
}

def frac_label(f: float) -> str:
    return f'{int(round(f * 100))}%'

def best_row(history):
    """Return epoch dict with highest val_dice."""
    return max(history, key=lambda r: r['val_dice'])

# Build per-fraction epoch axes once
epochs_per_frac = {f: [r['epoch'] for r in h] for f, h in results.items()}
print('Loaded fractions:', [frac_label(f) for f in sorted(results)])

## Plot 1 — Validation Dice learning curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for frac in sorted(results):
    hist = results[frac]
    ep   = epochs_per_frac[frac]
    vals = [r['val_dice'] for r in hist]
    ax.plot(ep, vals, color=FRAC_COLORS[frac], linewidth=2, label=frac_label(frac))
    best = best_row(hist)
    ax.scatter(best['epoch'], best['val_dice'],
               color=FRAC_COLORS[frac], s=80, zorder=5, edgecolors='k', linewidths=0.6)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Validation Dice', fontsize=12)
ax.set_title(f'fbsa_skip — Val Dice by Training Data Fraction', fontsize=13)
ax.legend(title='Train fraction', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(Path(OUT_DIR) / 'plot_val_dice_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Plot 2 — Training Dice learning curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for frac in sorted(results):
    hist = results[frac]
    ep   = epochs_per_frac[frac]
    ax.plot(ep, [r['train_dice'] for r in hist],
            color=FRAC_COLORS[frac], linewidth=2, label=frac_label(frac))

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Train Dice', fontsize=12)
ax.set_title('fbsa_skip — Train Dice by Training Data Fraction', fontsize=13)
ax.legend(title='Train fraction', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(Path(OUT_DIR) / 'plot_train_dice_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Plot 3 — Loss convergence curves (train + val)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for frac in sorted(results):
    hist  = results[frac]
    ep    = epochs_per_frac[frac]
    color = FRAC_COLORS[frac]
    label = frac_label(frac)
    axes[0].plot(ep, [r['train_loss'] for r in hist], color=color, linewidth=2, label=label)
    axes[1].plot(ep, [r['val_loss']   for r in hist], color=color, linewidth=2, label=label)

for ax, title in zip(axes, ['Train Loss (DiceBCE)', 'Val Loss (DiceBCE)']):
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title(f'fbsa_skip — {title}', fontsize=13)
    ax.legend(title='Train fraction', fontsize=10)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(Path(OUT_DIR) / 'plot_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Plot 4 — Best Val Dice vs data fraction

In [ ]:
fracs_sorted = sorted(results)
best_dices   = [best_row(results[f])['val_dice'] for f in fracs_sorted]
labels       = [frac_label(f) for f in fracs_sorted]
colors       = [FRAC_COLORS[f] for f in fracs_sorted]
x            = np.arange(len(fracs_sorted))

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(x, best_dices, color=colors, edgecolor='k', linewidth=0.6, width=0.55, zorder=3)
ax.plot(x, best_dices, 'o--', color='#333333', linewidth=1.5, markersize=7, zorder=4)

for bar, val in zip(bars, best_dices):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)
ax.set_xlabel('Training Data Fraction', fontsize=12)
ax.set_ylabel('Best Validation Dice', fontsize=12)
ax.set_title('fbsa_skip — Label Efficiency: Best Val Dice', fontsize=13)
ax.set_ylim(0, min(1.0, max(best_dices) * 1.12))
ax.grid(axis='y', alpha=0.3, zorder=0)
plt.tight_layout()
plt.savefig(Path(OUT_DIR) / 'plot_best_dice_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## Plot 5 — Multi-metric grid at convergence (Dice, IoU, Hausdorff)

In [ ]:
metrics_cfg = [
    ('val_dice',  'Best Val Dice',           True),   # (key, title, higher_is_better)
    ('val_iou',   'Best Val IoU',            True),
    ('val_hd',    'Val Hausdorff (px) @ best Dice epoch', False),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (key, title, hib) in zip(axes, metrics_cfg):
    values = [best_row(results[f])[key] for f in fracs_sorted]
    bars   = ax.bar(x, values, color=colors, edgecolor='k', linewidth=0.6,
                    width=0.55, zorder=3)
    ax.plot(x, values, 'o--', color='#333333', linewidth=1.5, markersize=6, zorder=4)

    fmt = '.4f' if key != 'val_hd' else '.1f'
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, val * 1.01,
                f'{val:{fmt}}', ha='center', va='bottom', fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=11)
    ax.set_xlabel('Training Data Fraction', fontsize=11)
    ax.set_title(f'fbsa_skip\n{title}', fontsize=12)
    ax.grid(axis='y', alpha=0.3, zorder=0)
    if key != 'val_hd':
        ax.set_ylim(0, min(1.0, max(values) * 1.15))

plt.suptitle('fbsa_skip — Label Efficiency: Metrics at Best Val-Dice Epoch',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(Path(OUT_DIR) / 'plot_multi_metric_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## Plot 6 — Val Dice + Val IoU side-by-side learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for frac in sorted(results):
    hist  = results[frac]
    ep    = epochs_per_frac[frac]
    color = FRAC_COLORS[frac]
    label = frac_label(frac)
    axes[0].plot(ep, [r['val_dice'] for r in hist], color=color, linewidth=2, label=label)
    axes[1].plot(ep, [r['val_iou']  for r in hist], color=color, linewidth=2, label=label)

for ax, metric in zip(axes, ['Val Dice', 'Val IoU']):
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel(metric, fontsize=12)
    ax.set_title(f'fbsa_skip — {metric}', fontsize=13)
    ax.legend(title='Train fraction', fontsize=10)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(Path(OUT_DIR) / 'plot_val_dice_iou_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary table

In [ ]:
import csv
import os
from tumor_seg.label_efficiency import summarise

rows = summarise(results)

# Print table
header = f"{'Frac':>5}  {'Best Epoch':>10}  {'Val Dice':>9}  {'Val IoU':>8}  {'Val HD (px)':>11}  {'Train Dice':>10}"
print(header)
print('-' * len(header))
for r in rows:
    print(f"{r['pct']:>5}  {r['best_epoch']:>10}  {r['val_dice']:>9.4f}  "
          f"{r['val_iou']:>8.4f}  {r['val_hd']:>11.2f}  {r['train_dice_at_best']:>10.4f}")

# Save CSV
csv_path = Path(OUT_DIR) / 'label_efficiency_summary.csv'
os.makedirs(OUT_DIR, exist_ok=True)
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)
print(f'\nSummary saved to {csv_path}')

## Plot 7 — Validation Hausdorff Distance curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for frac in sorted(results):
    hist = results[frac]
    ep   = epochs_per_frac[frac]
    ax.plot(ep, [r['val_hd'] for r in hist],
            color=FRAC_COLORS[frac], linewidth=2, label=frac_label(frac))

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Validation Hausdorff Distance (px)', fontsize=12)
ax.set_title('fbsa_skip — Val Hausdorff by Training Data Fraction', fontsize=13)
ax.legend(title='Train fraction', fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(Path(OUT_DIR) / 'plot_val_hd_curves.png', dpi=150, bbox_inches='tight')
plt.show()